# RAG Pipeline Flow 
This notebook walks through every stage of the RAG pipeline in order.
Each cell maps directly to one file in the `indexing/` or `generation/` folder.

**Pipeline:**
```
INDEXING                          GENERATION
────────────────────────────      ──────────────────────────
1. Load     sample_news.json      5. Embed query
2. Chunk    split into pieces     6. Retrieve  vector search
3. Embed    text → vectors        7. Generate  LLM + context
4. Store    vectors → Qdrant      8. Response  citations
```

In [7]:
import json, os 
from dotenv import load_dotenv
from datetime import datetime # parsing date time

load_dotenv()  # reads from .env

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"
print("✓ API key loaded")

✓ API key loaded


# Indexing Pipeline

## Stage 1: Load documents
`indexing/loaders/file_loader.py`

Read raw documents and outputs a list of `documents`

In [ ]:
class RawDocument:
    def __init__(self, id, title, text, url, publish_date, category="", tags=None):
        self.id = id
        self.title = title
        self.text = text
        self.url = url
        self.published = publish_date
        self.category = category
        self.tags = tags if tags is not None else []
        
def load(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
        
    docs = []
    for doc in raw: 
        docs.append(RawDocument(
            id=doc["id"],
            title=doc["title"],
            text=doc["content"],
            url=doc["url"],
            publish_date=datetime.fromisoformat(doc["publish_date"]),
            category=doc.get("category", ""),
            tags=doc.get("tags", []),
        ))
        
    return docs
        
docs = load("./data/news.json")

print(f"Loaded {len(docs)} documents\n")
for doc in docs:
    print(f" [{doc.id}] {doc.title}")

Loaded 11 documents

 [20260203_1] 樂團「微醺開根」首發專輯 唱當代族群故事
 [20260203_2] 排灣童聲唱出部落記憶 春日國小推《春迴》專輯
 [20260203_3] 仁愛鄉梅子白蘭地風味絕佳 奪國際一星獎章
 [20260203_4] 復興推「部落社區再復興」 凝共識促永續發展
 [20260203_5] 陳義信用行動感謝 職棒生涯文物捐贈退輔會
 [20260203_6] 台北國際電玩展 設計商「赴原Saikin」展新作
 [20260203_7] 嘉義文化科技創意賽頒獎 近50組團隊參賽
 [20260203_8] 《村裡遇見熊》書展亮相 記錄人熊共生故事錄
 [20260203_9] 台北國際書展登場 原民館推講座、交流活動
 [20260203_11] 首批承領人正式取得 大南澳土地所有權狀
 [20260203_12] 冒用縣府名義詐捐 花蓮縣府急澄清籲防詐


# Stage 2: Chunk
`indexing/chunkers/semantic_chunker.py`

Split each document into smaller pieces. 

**Why**: LLMs have content limits. Smaller chunks = more precise retrieval

Key rule: every chunk must keep `title`, `url`, `publish_date` for citation later.